# 02 — Independence gates and closure scans

Implements gates 1–3 of the pre-registered decision rule (`README.md`) on the
**even-parity half**: factorization-fit goodness of fit per process and total under
the quirk prescriptions; effective-count health; closure R = A/(B·C/D) along
boundary and event-cut scan trajectories with bootstrap covariance; guard-band
variants; the mJJ-sculpting diagnosis; and the non-closure systematic per plane.
Results are written to `gates_even.json` for notebooks 03/04.

In [ ]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

sys.path.insert(0, os.getcwd())
import abcd_tools as at
import study_setup as ss

hep.style.use("CMS")
plt.rcParams["figure.figsize"] = (7, 5)
LUMI_LABEL = r"59.8 fb$^{-1}$ (13 TeV, 2018 sim.)"

def cms_label(ax=None):
    hep.cms.label("Work in progress", data=False, rlabel=LUMI_LABEL, ax=ax)

# pre-skim / pre-filter sums of gen weights (see README "Normalization")
SUMW_PRE = {}
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_2MU2E))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_4MU))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_BKG_UNSKIMMED))

# Background sums built ONE SAMPLE AT A TIME (holding all 44 samples in memory OOMs
# the interactive node); signals are loaded lazily where needed, one at a time.
# TTJets kept at the campaign 471.7 pb; the NNLO alternative is an explicit rescale.
total_bkg, by_process = ss.accumulate_normalized(list(ss.BACKGROUNDS), SUMW_PRE)
def load_sig(s):
    return ss.load_normalized(s, SUMW_PRE)[0]
print(f"accumulated {len(ss.BACKGROUNDS)} backgrounds; scan hists: {len(total_bkg)}")

## Gate 1 — factorization fits

Fit μᵢⱼ = aᵢ·bⱼ on an adaptively rebinned grid (n_eff ≥ 10 per super-bin) for each
plane × process × quirk prescription. How to read it: p > 0.05 everywhere = the
background factorizes = ABCD's core assumption holds; p ≈ 1.0 rows are treated as
variance over-inflation, not passes.

In [ ]:
gate1 = {}
for ch in ss.PLANES:
    for pname in ss.PLANES[ch]:
        for presc in ["i", "ii"]:
            key = f"{ch}/{pname}/{presc}"
            ps = {}
            for p in ss.PROCESSES + ["total"]:
                hset = total_bkg if p == "total" else by_process[p]
                vals, var, xe, ye = ss.plane_arrays(hset, ch, pname, parity=0, prescription=presc)
                if presc == "ii":  # drop sentinel bins on iso plane axes too
                    spec = ss.PLANES[ch][pname]
                    if "iso" in spec["x"] and xe[0] < 0: vals, var, xe = vals[1:], var[1:], xe[1:]
                    if "iso" in spec["y"] and ye[0] < 0: vals, var, ye = vals[:, 1:], var[:, 1:], ye[1:]
                fit = at.factorization_fit(vals, var) if vals.sum() > 0 else {"pvalue": np.nan, "chi2": np.nan, "ndf": 0}
                ps[p] = fit["pvalue"]
            gate1[key] = ps
            print(f"{key:34s} " + " ".join(f"{p}:{v:7.3f}" for p, v in ps.items()))

## Gate 2 — statistical health at the working point

n_eff = (Σw)²/Σw² in each region at the SR working point. Regions with n_eff < 10
make the B·C/D error propagation anticonservative (demonstrated in
`test_abcd_tools.py`) — they fail the gate.

In [ ]:
gate2 = {}
for ch in ss.PLANES:
    for pname, spec in ss.PLANES[ch].items():
        vals, var, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
        reg = at.region_sums(vals, var, xe, ye, spec["xspec"], spec["yspec"])
        ne = {k: at.n_eff(*reg[k]) for k in "ABCD"}
        r, vr = at.closure_ratio(reg)
        gate2[f"{ch}/{pname}"] = {"n_eff": ne, "R": r, "R_err": np.sqrt(vr) if vr > 0 else np.nan,
                                  "yields": {k: reg[k][0] for k in "ABCD"}}
        flag = "" if min(ne.values()) >= 10 else "  <-- FAILS gate 2"
        print(f"{ch}/{pname:18s} A={reg['A'][0]:9.3g} R={r:6.3f}+-{np.sqrt(max(vr,0)):5.3f} "
              f"min_neff={min(ne.values()):7.1f}{flag}")

## Gate 3 — closure trends (loose → tight)

Boundary scans (R vs the two plane cuts) and event-cut scans (R vs the |Δφ| and mJJ
thresholds), with scan-point covariance from the per-bin bootstrap. The systematic is
the fitted trend extrapolated to the SR point ⊕ fit error, restricted to points with
n_eff(B, C, D) > 10. How to read the maps: color = R; hatched/blank = statistically
unhealthy; a flat map near 1 is what a good plane looks like.

In [ ]:
def boundary_scan(hists, ch, pname, xcuts, ycuts):
    spec = ss.PLANES[ch][pname]
    vals, var, xe, ye = ss.plane_arrays(hists, ch, pname, parity=0)
    R = np.full((len(xcuts), len(ycuts)), np.nan)
    ok = np.zeros_like(R, bool)
    for i, xc in enumerate(xcuts):
        for j, yc in enumerate(ycuts):
            reg = at.region_sums(vals, var, xe, ye, (spec["xspec"][0], xc), (spec["yspec"][0], yc))
            r, vr = at.closure_ratio(reg)
            R[i, j] = r
            ok[i, j] = min(at.n_eff(*reg[k]) for k in "ABCD") >= 10
    return R, ok

# incumbent planes: iso boundary grids on the 0.025 lattice
iso_cuts = [round(0.025 * k, 3) for k in range(2, 21)]
for ch, pname in [("2mu2e", "P1_iso_iso"), ("4mu", "Q1_iso_iso")]:
    R, ok = boundary_scan(total_bkg, ch, pname, iso_cuts, iso_cuts)
    fig, ax = plt.subplots()
    m = ax.pcolormesh(iso_cuts, iso_cuts, np.where(ok, R, np.nan).T, vmin=0.5, vmax=1.5, cmap="RdBu_r")
    fig.colorbar(m, ax=ax, label="R = A / (BC/D)")
    spec = ss.PLANES[ch][pname]
    ax.axvline(spec["xspec"][1], color="k", ls="--"); ax.axhline(spec["yspec"][1], color="k", ls="--")
    ax.set_xlabel(f"{spec['x']} boundary"); ax.set_ylabel(f"{spec['y']} boundary")
    ax.set_title(f"{ch} {pname} closure map (even half)", fontsize=12)
    cms_label(ax); plt.show()

In [ ]:
# event-cut scans with bootstrap covariance: R vs dphi threshold and vs mJJ threshold
def event_cut_scan(ch, pname, axis, thresholds):
    spec = ss.PLANES[ch][pname]
    pts, Rs, errs = [], [], []
    h = at.get_channel(total_bkg[spec["hist"]], ss.CHANNELS[ch])
    for t in thresholds:
        sel = dict(spec["cuts"]); sel[axis] = ("ge", float(t)); sel["parity"] = ("bin", 0)
        vals, var, xe, ye = at.project_plane(h, spec["x"], spec["y"], sel)
        reg = at.region_sums(vals, var, xe, ye, spec["xspec"], spec["yspec"])
        if min(at.n_eff(*reg[k]) for k in "ABCD") < 10:
            continue
        r, vr = at.closure_ratio(reg)
        pts.append(t); Rs.append(r); errs.append(np.sqrt(max(vr, 0)))
    return np.array(pts), np.array(Rs), np.array(errs)

gate3 = {}
for ch, pname in [("2mu2e", "P1_iso_iso"), ("2mu2e", "P2_muiso_dphi"), ("2mu2e", "P4_muiso_mjj"),
                  ("2mu2e", "P8_dphi_mjj"), ("4mu", "Q1_iso_iso"), ("4mu", "Q2_iso0_dphi"),
                  ("4mu", "Q5_dphi_mjj")]:
    spec = ss.PLANES[ch][pname]
    fig, ax = plt.subplots()
    syst = {}
    for axis, thr in [("dphi", [0.0, 0.8, 1.2, 1.6, 2.0]), ("mjj", [0, 50, 100, 150])]:
        if axis in (spec["x"], spec["y"]) or axis not in spec["cuts"]:
            continue
        t, R, e = event_cut_scan(ch, pname, axis, thr)
        if len(t) < 2:
            syst[axis] = None; continue
        ax.errorbar(t, R, e, marker="o", ls="-", capsize=2, label=f"{axis} scan")
        # linear trend extrapolated to the SR threshold
        w = 1 / e**2
        coef = np.polyfit(t, R, 1, w=w)
        r_sr = np.polyval(coef, spec["cuts"][axis][1])
        syst[axis] = abs(r_sr - 1.0)
    ax.axhline(1, color="gray", lw=1)
    ax.set_xlabel("event-cut threshold"); ax.set_ylabel("R")
    ax.set_title(f"{ch} {pname}", fontsize=12); ax.legend(); cms_label(ax); plt.show()
    gate3[f"{ch}/{pname}"] = syst
    print(f"{ch}/{pname}: non-closure estimates {syst}")

## Guard bands and quirk prescriptions

The same closure evaluated with 1–2 guard bins around each boundary (the user's
"gaps" idea) and under prescriptions (i)/(ii)/(iii). A verdict that flips under these
variations is not robust.

In [ ]:
for ch, pname in [("2mu2e", "P1_iso_iso"), ("4mu", "Q1_iso_iso")]:
    spec = ss.PLANES[ch][pname]
    vals, var, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
    print(f"--- {ch}/{pname}")
    for label, kw in [("nominal (i)", {}),
                      ("guard 1 bin", dict(xguard=1, yguard=1)),
                      ("guard 2 bins", dict(xguard=2, yguard=2)),
                      ("sentinel excluded (iii)", dict(xlo=0.0, ylo=0.0))]:
        reg = at.region_sums(vals, var, xe, ye, spec["xspec"], spec["yspec"], **kw)
        r, vr = at.closure_ratio(reg)
        print(f"  {label:24s} R = {r:6.3f} +- {np.sqrt(max(vr,0)):5.3f}  A = {reg['A'][0]:9.4g}")

## mJJ sculpting diagnosis + κ + round-2 validation region

Closure and factorization split by mJJ region, per process — the head-on version of
the sculpting question. The low-mJJ region is the designated round-2 (data) validation
region if it stays signal-depleted (checked in 03).

In [ ]:
for ch, pname in [("2mu2e", "P1_iso_iso"), ("4mu", "Q1_iso_iso")]:
    spec = ss.PLANES[ch][pname]
    h = at.get_channel(total_bkg[spec["hist"]], ss.CHANNELS[ch])
    print(f"--- {ch}/{pname}")
    for lab, mjjsel in [("mJJ >= 150 (SR)", ("ge", 150.0)), ("mJJ < 150 (VR)", ("window", 0.0, 150.0))]:
        sel = dict(spec["cuts"]); sel["mjj"] = mjjsel; sel["parity"] = ("bin", 0)
        vals, var, xe, ye = at.project_plane(h, spec["x"], spec["y"], sel)
        reg = at.region_sums(vals, var, xe, ye, spec["xspec"], spec["yspec"])
        r, vr = at.closure_ratio(reg)
        k, vk = at.kappa(reg)
        corr = at.weighted_correlation(vals, xe, ye)
        print(f"  {lab:18s} R = {r:6.3f}+-{np.sqrt(max(vr,0)):5.3f}  kappa = {k:6.3f}  r_w = {corr:6.3f}")

In [ ]:
# persist gate results for notebooks 03/04
out = {"gate1": gate1, "gate2": {k: {"n_eff": v["n_eff"], "R": v["R"], "R_err": v["R_err"],
                                     "yields": v["yields"]} for k, v in gate2.items()},
       "gate3_nonclosure": gate3}
json.dump(out, open(os.path.join(ss.WORKDIR, "gates_even.json"), "w"), indent=1, default=float)
print("wrote gates_even.json")